# 06-P2. paper-aligned autoencoder

이 노트북은 06-P1에서 선택한 정상 행동 구간만으로 Autoencoder baseline을 학습하고 reconstruction error 기반 anomaly score를 계산한다.

현재 구현 제약:
- 프로젝트 환경에는 TensorFlow / PyTorch가 없다.
- 따라서 baseline은 `StandardScaler + sklearn MLPRegressor 재구성형 autoencoder`로 고정한다.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml을 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
WINDOW_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
FEATURE_COLUMNS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_features' / 'feature_columns.csv'
IMPUTATION_VALUES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_features' / 'imputation_values.csv'
NORMAL_SELECTION_PATH = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned' / 'normal_behaviour_training_windows.csv'
EVENT_EVAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned' / 'event_evaluation_windows.csv'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'paper_aligned'
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SCORES_OUTPUT = OUTPUT_DIR / 'autoencoder_reconstruction_scores.csv'
THRESHOLDS_OUTPUT = OUTPUT_DIR / 'autoencoder_thresholds.csv'
MODEL_OUTPUT = MODEL_DIR / 'paper_aligned_autoencoder_model.joblib'
SCALER_OUTPUT = MODEL_DIR / 'paper_aligned_autoencoder_scaler.joblib'
METADATA_OUTPUT = MODEL_DIR / 'autoencoder_metadata.json'

MODEL_VERSION = 'paper_aligned_autoencoder_v1'
HIDDEN_LAYER_SIZES = (64, 32, 64)
RANDOM_STATE = 42
THRESHOLD_QUANTILES = [0.95, 0.975, 0.99]

key_columns = ['manufacturer', 'substation_id', 'window_start', 'window_end', 'label']

feature_columns_df = pd.read_csv(FEATURE_COLUMNS_PATH)
feature_columns = feature_columns_df.loc[feature_columns_df['selected_for_baseline'] == True, 'column_name'].tolist()
imputation_df = pd.read_csv(IMPUTATION_VALUES_PATH)
imputation_map = dict(zip(imputation_df['column_name'], imputation_df['imputation_value']))

windows_df = pd.read_csv(WINDOW_PATH, parse_dates=['window_start', 'window_end'])
normal_selection_df = pd.read_csv(NORMAL_SELECTION_PATH, parse_dates=['window_start', 'window_end', 'event_start', 'event_end', 'report_date'])
event_eval_df = pd.read_csv(EVENT_EVAL_PATH, parse_dates=['window_start', 'window_end', 'event_start', 'event_end', 'report_date'])

base_df = event_eval_df.merge(
    windows_df[key_columns + feature_columns],
    on=key_columns,
    how='left',
    validate='one_to_one',
)

normal_flag_columns = [
    'selected_for_autoencoder_train',
    'selected_for_autoencoder_validation',
    'selected_for_normal_event_holdout',
    'selected_for_autoencoder_fit',
]

base_df = base_df.merge(
    normal_selection_df[key_columns + normal_flag_columns],
    on=key_columns,
    how='left',
    validate='one_to_one',
)

for column in normal_flag_columns:
    base_df[column] = base_df[column].fillna(False).astype(bool)

missing_feature_rows = int(base_df[feature_columns].isna().all(axis=1).sum())
if missing_feature_rows > 0:
    raise ValueError(f'feature merge 실패 row 수: {missing_feature_rows}')

X = base_df[feature_columns].copy()
for column in feature_columns:
    X[column] = X[column].fillna(imputation_map[column])
X = X.astype(float)

train_mask = base_df['selected_for_autoencoder_train'].astype(bool)
validation_mask = base_df['selected_for_autoencoder_validation'].astype(bool)

scaler = StandardScaler()
X_train = scaler.fit_transform(X.loc[train_mask])
X_validation = scaler.transform(X.loc[validation_mask])
X_all = scaler.transform(X)

autoencoder = MLPRegressor(
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
)
autoencoder.fit(X_train, X_train)

reconstructed_all = autoencoder.predict(X_all)
squared_error_all = (X_all - reconstructed_all) ** 2
rmse_all = np.sqrt(squared_error_all.mean(axis=1))
mse_all = squared_error_all.mean(axis=1)

train_rmse = rmse_all[train_mask.to_numpy()]
validation_rmse = rmse_all[validation_mask.to_numpy()]

threshold_records = []
threshold_map = {}
for quantile in THRESHOLD_QUANTILES:
    threshold_value = float(np.quantile(train_rmse, quantile))
    threshold_map[quantile] = threshold_value
    threshold_records.append(
        {
            'model_version': MODEL_VERSION,
            'threshold_name': f'train_rmse_p{str(quantile).replace('.', '')}',
            'quantile': quantile,
            'threshold_value': threshold_value,
            'train_above_count': int((train_rmse >= threshold_value).sum()),
            'validation_above_count': int((validation_rmse >= threshold_value).sum()),
        }
    )

train_p99 = threshold_map[0.99]
base_df['reconstruction_mse'] = mse_all
base_df['reconstruction_rmse'] = rmse_all
base_df['anomaly_score'] = base_df['reconstruction_rmse'] / train_p99
base_df['model_version'] = MODEL_VERSION
base_df['feature_count'] = len(feature_columns)

for quantile, threshold_value in threshold_map.items():
    suffix = str(quantile).replace('.', '')
    base_df[f'train_rmse_p{suffix}'] = threshold_value
    base_df[f'is_above_train_p{suffix}'] = base_df['reconstruction_rmse'] >= threshold_value

scores_output_columns = [
    'manufacturer',
    'substation_id',
    'window_start',
    'window_end',
    'label',
    'fault_label',
    'fault_event_id',
    'estimated_lead_time_hours',
    'configuration_type',
    'window_source_type',
    'split_time_based',
    'split_substation_based',
    'event_type',
    'event_id',
    'event_label',
    'event_start',
    'event_end',
    'report_date',
    'event_split',
    'selection_reason',
    'evaluation_reason',
    'selected_for_autoencoder_train',
    'selected_for_autoencoder_validation',
    'selected_for_normal_event_holdout',
    'selected_for_autoencoder_fit',
    'selected_for_event_eval',
    'selected_for_event_tuning',
    'selected_for_event_holdout',
    'reconstruction_mse',
    'reconstruction_rmse',
    'anomaly_score',
    'train_rmse_p095',
    'train_rmse_p0975',
    'train_rmse_p099',
    'is_above_train_p095',
    'is_above_train_p0975',
    'is_above_train_p099',
    'feature_count',
    'model_version',
]
scores_df = base_df[scores_output_columns].copy()
scores_df.to_csv(SCORES_OUTPUT, index=False, encoding='utf-8-sig')

thresholds_df = pd.DataFrame(threshold_records)
thresholds_df.to_csv(THRESHOLDS_OUTPUT, index=False, encoding='utf-8-sig')

joblib.dump(autoencoder, MODEL_OUTPUT)
joblib.dump(scaler, SCALER_OUTPUT)

metadata = {
    'model_version': MODEL_VERSION,
    'model_type': 'sklearn_mlp_autoencoder_regressor',
    'hidden_layer_sizes': list(HIDDEN_LAYER_SIZES),
    'random_state': RANDOM_STATE,
    'feature_count': len(feature_columns),
    'feature_columns': feature_columns,
    'train_row_count': int(train_mask.sum()),
    'validation_row_count': int(validation_mask.sum()),
    'scoring_row_count': int(len(base_df)),
    'train_rmse_mean': float(train_rmse.mean()),
    'validation_rmse_mean': float(validation_rmse.mean()),
    'train_rmse_thresholds': {
        f'p{str(quantile).replace('.', '')}': float(value)
        for quantile, value in threshold_map.items()
    },
    'note': 'anomaly_score is reconstruction_rmse normalised by train_rmse_p099',
}
METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

summary_df = pd.DataFrame(
    [
        {
            'subset': 'train_normal',
            'rows': int(train_mask.sum()),
            'rmse_mean': float(train_rmse.mean()),
            'rmse_p95': float(np.quantile(train_rmse, 0.95)),
            'rmse_p99': float(np.quantile(train_rmse, 0.99)),
        },
        {
            'subset': 'validation_normal',
            'rows': int(validation_mask.sum()),
            'rmse_mean': float(validation_rmse.mean()),
            'rmse_p95': float(np.quantile(validation_rmse, 0.95)),
            'rmse_p99': float(np.quantile(validation_rmse, 0.99)),
        },
    ]
)

display(summary_df)
display(thresholds_df)
display(
    scores_df.groupby(['event_type', 'event_split'])[['is_above_train_p099']]
    .sum()
    .reset_index()
)
print(f'saved: {SCORES_OUTPUT}')
print(f'saved: {THRESHOLDS_OUTPUT}')
print(f'saved: {MODEL_OUTPUT}')
print(f'saved: {SCALER_OUTPUT}')
print(f'saved: {METADATA_OUTPUT}')


C:\Users\Admin\AppData\Local\Temp\ipykernel_25368\314410801.py:137: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_df['reconstruction_mse'] = mse_all
C:\Users\Admin\AppData\Local\Temp\ipykernel_25368\314410801.py:138: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base_df['reconstruction_rmse'] = rmse_all
C:\Users\Admin\AppData\Local\Temp\ipykernel_25368\314410801.py:139: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider 

,subset,rows,rmse_mean,rmse_p95,rmse_p99
0,train_normal,1216,0.630803,1.117752,1.486695
1,validation_normal,269,0.645922,1.094318,1.488670


,model_version,threshold_name,quantile,threshold_value,train_above_count,validation_above_count
0,paper_aligned_autoencoder_v1,train_rmse_p095,0.950,1.117752,61,11
1,paper_aligned_autoencoder_v1,train_rmse_p0975,0.975,1.271971,31,6
2,paper_aligned_autoencoder_v1,train_rmse_p099,0.990,1.486695,13,3


,event_type,event_split,is_above_train_p099
0,fault_event,holdout,19
1,fault_event,train,155
2,fault_event,validation,37
3,normal_event,holdout,8
4,normal_event,train,24
5,normal_event,validation,3


saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\autoencoder_reconstruction_scores.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\autoencoder_thresholds.csv
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\models\paper_aligned_autoencoder_model.joblib
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\models\paper_aligned_autoencoder_scaler.joblib
saved: C:\Project3\HeatGrid_Agent\data\processed\paper_aligned\models\autoencoder_metadata.json


## 다음 단계

- `autoencoder_reconstruction_scores.csv`를 입력으로 06-P3 event-wise detection 평가를 수행한다.
- `is_above_train_p099`는 point anomaly 기준선일 뿐이며, 06-P3에서 criticality counter를 거친 뒤 최종 detection signal을 만든다.
